## Paso 0: Instalación y configuración

In [1]:
#!pip install -q pypdf gradio google-genai python-dotenv

In [2]:
from pypdf import PdfReader
import gradio as gr
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paso 1: Extracción de texto del PDF


In [3]:
def leer_pdf(ruta_pdf, optional_print=False):
    reader = PdfReader(ruta_pdf)

    if optional_print:
        num_paginas = len(reader.pages)
        print(f"{num_paginas} páginas encontradas en el PDF.")
        
    texto_completo = ""

    for pagina in reader.pages:
        texto = pagina.extract_text()
        if texto:
            texto_completo += texto + "\n"

  
    return texto_completo

In [4]:
ruta_pdf = "ia.pdf"
texto_documento = leer_pdf(ruta_pdf)

print(texto_documento[:100])
print("Total de caracteres:", len(texto_documento))

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and
Total de caracteres: 39630


## Paso 2: Inicialización del cliente de Gemini


In [5]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("La clave de API no se encontró.")

cliente = genai.Client(api_key=API_KEY);
MODELO = "gemini-2.5-flash-lite"
print("gemini cliente creado correctamente")


gemini cliente creado correctamente


## Paso 3: Función de chat con contexto del documento


In [6]:
def build_system_prompt(document_text):
    return f"""
    Eres un asistente virtual especializado en analizar documentos PDF. Tu tarea es responder solo con información del documento. Si la respuesta no se puede inferir del documento, dilo. No inventes información.

    Contenido del documento:
    {document_text}

    Pregunta del usuario:
    """

In [ ]:
def _to_text(value):
    """Convierte cualquier valor a texto plano (cadena)."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        text = value.get("text") or value.get("content")
        if text is not None:
            return _to_text(text)
        return ""
    if isinstance(value, list):
        parts = [_to_text(item) for item in value]
        return "".join(p for p in parts if p)
    return str(value)

def chat_con_documento(message, history, document_text):

    conversation_memory = []

    # Procesar historial (formato dict o tupla)
    for entry in history:
        if isinstance(entry, dict):
            raw_role = entry.get("role", "user")
            role = "model" if raw_role in ("assistant", "model") else "user"
            content_text = _to_text(entry.get("content", ""))
            if content_text:
                conversation_memory.append(
                    types.Content(role=role, parts=[types.Part(text=content_text)])
                )
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            user_text = _to_text(entry[0])
            assistant_text = _to_text(entry[1])
            if user_text:
                conversation_memory.append(
                    types.Content(role="user", parts=[types.Part(text=user_text)])
                )
            if assistant_text:
                conversation_memory.append(
                    types.Content(role="model", parts=[types.Part(text=assistant_text)])
                )

    # Agregar mensaje actual del usuario
    message_text = _to_text(message)
    if message_text:
        conversation_memory.append(
            types.Content(role="user", parts=[types.Part(text=message_text)])
        )

    # Llamada a la API — acumular TODO el stream antes de retornar
    try:
        stream = cliente.models.generate_content_stream(
            model=MODELO,
            config=types.GenerateContentConfig(
                system_instruction=build_system_prompt(document_text),
            ),
            contents=conversation_memory,
        )

        # Acumular la respuesta COMPLETA primero
        respuesta_completa = ""
        for chunk in stream:
            delta = getattr(chunk, "text", None)
            if delta:
                respuesta_completa += delta

        # Guardar respuesta en memoria
        conversation_memory.append(
            types.Content(role="model", parts=[types.Part(text=respuesta_completa)])
        )

        # Retornar la respuesta completa de una sola vez
        return respuesta_completa

    except Exception as e:
        return f"Error: {e}"


## Paso 4: Interfaz Gradio


In [ ]:
texto_pdf = leer_pdf(ruta_pdf, 1)
print(f"Longitud del texto: {len(texto_pdf)}")

Longitud del texto: 39630


In [ ]:
# Interfaz Gradio

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Chat con tu PDF (RAG básico sin embeddings)",
    description=(
        "Haz preguntas sobre el documento cargado. El asistente debe responder solo con información del PDF; "
        "si no está en el texto, debe decirlo explícitamente."
    ),
    additional_inputs=[gr.Textbox(value=texto_documento, visible=False)],
)

demo.launch(server_name="0.0.0.0", server_port=8080)



* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.



=== conversation_memory (2 turnos) ===
  [0] USER: '¿Qué es GPT-4?'
  [1] MODEL: 'El documento no menciona GPT-4.'

=== conversation_memory (4 turnos) ===
  [0] USER: '¿Qué es GPT-4?'
  [1] MODEL: 'El documento no menciona GPT-4.'
  [2] USER: '¿Cuál es la arquitectura principal propuesta en el paper?'
  [3] MODEL: 'La arquitectura principal propuesta en el paper es el **Transformer**.'

=== conversation_memory (6 turnos) ===
  [0] USER: '¿Qué es GPT-4?'
  [1] MODEL: 'El documento no menciona GPT-4.'
  [2] USER: '¿Cuál es la arquitectura principal propuesta en el paper?'
  [3] MODEL: 'La arquitectura principal propuesta en el paper es el **Transformer**.'
  [4] USER: '¿Qué es el mecanismo de atención?'
  [5] MODEL: 'El mecanismo de atención se describe como una función que mapea una consulta (query) y un conjunto de pares clave-valor ...'

=== conversation_memory (8 turnos) ===
  [0] USER: '¿Qué es GPT-4?'
  [1] MODEL: 'El documento no menciona GPT-4.'
  [2] USER: '¿Cuál es la arquitec

## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.

¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 
**Respuesta:** El modelo sí se ciñó al documento porque, ante la pregunta trampa (“¿Qué es GPT-4?”), no intentó definirlo con conocimiento general. En su lugar, indicó que esa información no aparece en el PDF (o que no puede inferirse del texto).
Eso muestra que respetó la instrucción del system prompt: responder solo con contenido del documento y evitar inventar datos externos.


## Reflexión final
 
Discutan en grupo:
 
1. **¿Cuál es la limitación principal de este enfoque?**
   Piensen en qué pasaría con un documento de 1000 páginas.

   **Respuesta:** La principal limitación es la escalabilidad. En un documento de 1000 páginas:
   - Enviaríamos millones de caracteres en cada request (consumiendo muchos tokens)
   - Aumentaría significativamente la latencia de respuesta
   - Sería muy costoso en términos de API calls
   - El modelo podría tener dificultad para procesar contextos tan extensos
   - No es práctico para aplicaciones con múltiples documentos grandes
 
2. **¿Por qué existe RAG?**
   ¿Cómo resolvería RAG el problema que identificaron?
   
   **Respuesta:** RAG existe precisamente para resolver este problema. En lugar de inyectar el documento completo, RAG:
   - Divide el documento en fragmentos (chunks) más pequeños
   - Crea embeddings (representaciones vectoriales) de cada fragmento
   - Cuando el usuario hace una pregunta, busca los fragmentos más relevantes usando similitud vectorial
   - Inyecta solo esos fragmentos específicos en el context del modelo
   - Resultado: mucho menos overhead de tokens, respuestas más rápidas, aplicable a documentos gigantes
   
3. **¿Qué información podría "filtrarse" aunque el system prompt diga que no?**
   El modelo tiene conocimiento previo sobre el paper — ¿cómo verificarían
   que está respondiendo desde el documento y no desde su entrenamiento?

   **Respuesta:** El modelo podría usar su conocimiento previo entrenado sobre el paper, en lugar de limitarse solo al documento inyectado. Para verificar esto:
   - Pedir citas textuales: Modificar el system prompt para que el modelo cite la sección o página exacta de dónde extrae cada información
   - Hacer preguntas trampa: Incluir información falsa o ficción dentro del mensaje de usuario ("¿En qué página el paper menciona que Transformers usan RNNs?") — si el modelo cae, está usando conocimiento externo
   - Preguntas de detalles obscuros: Hacer preguntas sobre números específicos, definiciones precisas o secciones menores que el modelo probablemente no conoce bien de su entrenamiento
   - Comparar respuestas: Usar un modelo con y sin el documento inyectado para ver si hay diferencias significativas
